# Xiaohongshu Parser Test

Platform: xiaohongshu.com / xhslink.com

Strategy: Playwright + system Chrome (headful) + stealth

Output: Markdown

In [ ]:
TEST_URL = "https://www.xiaohongshu.com/explore/example"  # Change to real xhs URL
import os

OUTPUT_DIR = os.path.join(os.getcwd(), "test-output")

In [ ]:
import os
chrome_paths = [
    r"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe",
    r"C:\\Program Files (x86)\\Google\\Chrome\\Application\\chrome.exe",
    os.path.expanduser(r"~\\AppData\\Local\\Google\\Chrome\\Application\\chrome.exe"),
]
chrome_found = next((p for p in chrome_paths if os.path.exists(p)), None)
print(f"Chrome: {chrome_found or 'NOT FOUND'}")

In [ ]:
import re
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def extract_xiaohongshu_data(html: str):
    soup = BeautifulSoup(html, "html.parser")
    result = {"title": "", "author": "", "content_html": "", "method": ""}

    # L1: OG meta
    og_title = soup.find("meta", property="og:title")
    og_desc = soup.find("meta", property="og:description")
    result["title"] = og_title["content"] if og_title else ""

    # L2: JSON-LD
    ld_script = soup.find("script", type="application/ld+json")
    if ld_script:
        try:
            import json
            ld = json.loads(ld_script.string)
            if isinstance(ld, list): ld = ld[0]
            result["title"] = result["title"] or ld.get("headline", "")
            result["author"] = ld.get("author", {}).get("name", "")
            result["content_html"] = f"<p>{ld.get('description', '')}</p>"
            result["method"] = "json-ld"
            return result
        except: pass

    # L3: DOM selectors (note content)
    content = soup.select_one(".note-content, .content, .desc" )
    if content:
        result["content_html"] = str(content)
        result["method"] = "DOM-selector"

    # L4: OG fallback
    if not result["content_html"] and og_desc:
        result["content_html"] = f"<p>{og_desc['content']}</p>"
        result["method"] = "og-extract"

    return result

def html_to_markdown(content_html: str, title: str = "", author: str = "") -> str:
    soup = BeautifulSoup(content_html, "html.parser")
    for img in soup.find_all("img"):
        actual = img.get("data-src") or img.get("data-original")
        if actual and not img.get("src"):
            img["src"] = actual
    markdown = md(str(soup), heading_style="ATX", bullets="-").strip()
    parts = []
    if title: parts.append(f"# {title}")
    if author: parts.append(f"> Author: {author}")
    parts.append(markdown)
    return "".join(parts)

In [ ]:
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    launch_opts = {"headless": False, "args": ["--disable-blink-features=AutomationControlled"]}
    if chrome_found: launch_opts["channel"] = "chrome"
    browser = p.chromium.launch(**launch_opts)
    ctx = browser.new_context(viewport={"width": 1920, "height": 1080})
    ctx.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined});")
    page = ctx.new_page()
    page.goto(TEST_URL, wait_until="domcontentloaded", timeout=30000)
    page.wait_for_timeout(3000)
    html = page.content()
    browser.close()

print(f"Fetched HTML: {len(html)} chars")

In [ ]:
data = extract_xiaohongshu_data(html)
markdown = html_to_markdown(data['content_html'], data['title'], data['author'])
plaintext = re.sub(r'<[^>]+>', ' ', data['content_html']).replace("  ", " ").strip()

print(f"Method: {data['method']}")
print(f"Title: {data['title']}")
print(f"Author: {data['author']}")
print(f"Markdown: {len(markdown)} chars")

In [ ]:
from IPython.display import Markdown as IPMarkdown, display
preview = markdown[:4000] + ("... (truncated)" if len(markdown) > 4000 else "")
display(IPMarkdown(preview))

In [ ]:
import os
from urllib.parse import urlparse

os.makedirs(OUTPUT_DIR, exist_ok=True)
slug = urlparse(TEST_URL).path.strip("/").replace("/", "-") or "page"

with open(os.path.join(OUTPUT_DIR, f"{slug}.html"), "w", encoding="utf-8") as f: f.write(html)
with open(os.path.join(OUTPUT_DIR, f"{slug}.md"), "w", encoding="utf-8") as f: f.write(markdown)
with open(os.path.join(OUTPUT_DIR, f"{slug}.txt"), "w", encoding="utf-8") as f: f.write(plaintext)
print("Saved to test-output/")